#Imports

In [0]:
%run ./common

In [0]:
from pyspark.sql import Window

#Ingest and Cleanup

##bls

In [0]:
bls_df = spark.read.csv("s3://rearc-quest-107628756615-us-east-2-an/bls-data/pr/pr.data.0.Current", sep="\t", header=True)
bls_clean_df = clean_df(bls_df)
bls_clean_typed_df = bls_clean_df.withColumn("value", f.col("value").cast(t.DoubleType()))

##dataus

In [0]:
json_df = spark.read.json("s3://rearc-quest-107628756615-us-east-2-an/datausa/annual_us_pop_2013_thru_2024.json")
dataus_df = json_df.select(f.explode("data")).select("col.*")
dataus_clean_df = clean_df(dataus_df)

#Analyze

##Mean and STD US Pop 2013 through 2018

1. Filter to relevant years
2. Group whole dataset
3. Agg mean and standard deviation

In [0]:
mean_std_pop = dataus_clean_df.filter(f.col('year').between(2013, 2018))\
    .groupBy()\
    .agg(
        f.mean('population').alias('mean_pop'),
        f.stddev('population').alias('std_pop')
    )

mean_std_pop.display()

## Best total value by series_id and year

1. Group by series_id and year
2. Sum value to get total value by series_id and year
3. Partition by series_id and rank each year's total_value against other years within partition
4. Select rank = 1 to get the highest value year for each series_id
5. Remove rank column for reporting

In [0]:
best_year_in_series = bls_clean_df.groupBy('series_id','year')\
    .agg(f.sum('value').alias('total_value'))\
    .withColumn('rank', f.row_number().over(Window.partitionBy('series_id').orderBy(f.desc('total_value'))))\
    .filter(f.col('rank') == 1)\
    .select('series_id','year','total_value')

best_year_in_series.display()

##value and pop for series_id = PRS30006032 and period = Q01

1. filter bls data to series_id = PRS30006032 and period = Q01
2. left join dataus on year (keeps all bls records, adds Population "if available")
3. select reporting columns

In [0]:
bls_clean_typed_df.filter((f.col('series_id')=='PRS30006032')&(f.col('period')=='Q01'))\
    .join(dataus_clean_df, on=bls_clean_typed_df.year == dataus_clean_df.Year, how='left')\
    .select("series_id", bls_clean_typed_df.year, "period", "value", "Population").display()